# Day 5 · Notebook 5 of 5 – Genie space, runbook, presentation and sign-off

| Part | Content |
|---|---|
| 1 | Genie space setup (steps for the UI) |
| 2 | Fixes applied to earlier notebooks on Day 5 |
| 3 | Runbook |
| 4 | Key numbers and outline for the presentation |
| 5 | Day 5 evidence summary |
| 6 | Project sign-off |

**Run after** notebooks `7_1` to `7_4`. Only the cells in Parts 4 and 5 run code. They only read tables.

## Part 1 – Genie space setup

**Step 1 – Create the space.** Go to **Genie → New**. Name: `Equipment Reliability – Capstone 05`. Choose a running SQL warehouse (Serverless recommended). Share it with the `analysts` group. People can only query tables they have SELECT on.

**Step 2 – Add these tables and views** (*Add data*):

| Object | Why |
|---|---|
| `capstone_5iot.gold.vw_kpi_headline` | Headline KPIs, MTBF, MTTR |
| `capstone_5iot.gold.vw_daily_trend` | Daily trends |
| `capstone_5iot.gold.vw_machine_reliability` | Per-machine KPIs with plant, type, criticality |
| `capstone_5iot.gold.vw_plant_summary` | Per-plant and per-region KPIs |
| `capstone_5iot.gold.vw_failure_pareto` | Failure codes, categories, root causes, cumulative % |
| `capstone_5iot.gold.vw_failure_correlation` | Abnormal readings before failures |
| `capstone_5iot.gold.vw_maintenance` | Work-order status, cost, time to complete |
| `capstone_5iot.gold.vw_data_quality` | Rejects, gaps, test pass rate |
| `capstone_5iot.gold.vw_machine_failure_risk` | Pilot risk scores |
| `capstone_5iot.gold.mart_sensor_daily` | Machine-day grain for date filtering |
| `capstone_5iot.gold.fact_maintenance_episodes` | One row per failure event |
| `capstone_5iot.gold.dim_machine` | Machine attributes |
| `capstone_5iot.gold.dim_plant` | Plant attributes |
| `capstone_5iot.gold.dim_failure_code` | Failure-code attributes |

**Step 3 – General instructions.** Paste into *Instructions → Text*:

> You answer questions about manufacturing equipment reliability for 150 machines in 159 plants, with daily data from 2026-07-01 to 2026-08-26.
> - Availability per machine-day = (1440 - downtime_minutes) / 1440. For a group or period use (machines x days x 1440 - SUM(downtime_minutes)) / (machines x days x 1440). Do not average daily percentages.
> - Abnormal reading = temperature_c > 85 OR vibration_mm_s > 7.1 OR pressure_bar > 18. Abnormal rate = SUM(abnormal_count) / SUM(reading_count).
> - Total downtime is 3,349 minutes from 665 failures. Downtime comes from fact_maintenance_episodes.downtime_minutes.
> - MTBF hours = (machines x days x 24 - downtime hours) / failures. MTTR = downtime minutes / failures.
> - Join facts to dimensions on machine_sk, plant_sk, failure_code_sk and date_key (yyyyMMdd integer).
> - "Last N days" means relative to the latest activity_date in the data (2026-08-26), not today.
> - Failure events only cover 2026-07-01 to 2026-07-02. Say so when a question asks for failure trends over time.
> - Risk scores in vw_machine_failure_risk come from a pilot model with near-chance accuracy. Present them as a relative ranking and mention this.
> - Round percentages to 2 decimals and show minutes as whole numbers.

**Step 4 – Sample questions** (*Settings → Sample questions*):
1. What is the overall availability and total downtime for the period?
2. Which 5 machines had the most downtime, and which plants are they in?
3. Which failure codes account for 80% of downtime?
4. Show the daily abnormal reading rate as a line chart.
5. Which region has the lowest availability?
6. How do HIGH criticality machines compare with LOW on failures and downtime?
7. How many work orders are still open, and what parts cost do they carry?
8. How many records were quarantined, and for which DQ rules?

**Step 5 – Trusted SQL queries** (*Instructions → SQL queries*, mark them as trusted):

```sql
-- 1. Availability and downtime by region (weighted formula)
SELECT p.region,
       COUNT(DISTINCT m.machine_id) AS machines,
       SUM(f.downtime_minutes)      AS downtime_minutes,
       ROUND(1 - SUM(f.downtime_minutes) /
             (COUNT(DISTINCT m.machine_id) * (SELECT COUNT(*) FROM capstone_5iot.gold.dim_date) * 1440.0), 6) AS availability
FROM capstone_5iot.gold.dim_machine m
JOIN capstone_5iot.gold.dim_plant p ON m.plant_sk = p.plant_sk
LEFT JOIN capstone_5iot.gold.fact_maintenance_episodes f ON f.machine_sk = m.machine_sk
GROUP BY p.region
ORDER BY downtime_minutes DESC;
```

```sql
-- 2. Top N machines by downtime
SELECT machine_id, plant_name, machine_type, criticality, failures, downtime_minutes,
       ROUND(abnormal_reading_rate * 100, 2) AS abnormal_pct, availability, downtime_rank
FROM capstone_5iot.gold.vw_machine_reliability
WHERE downtime_rank <= :top_n
ORDER BY downtime_minutes DESC, machine_id;
```

```sql
-- 3. Daily abnormal reading rate for a date range (weighted, not averaged)
SELECT activity_date,
       SUM(abnormal_count)                                AS abnormal_readings,
       SUM(reading_count)                                 AS valid_readings,
       ROUND(SUM(abnormal_count) / SUM(reading_count), 4) AS abnormal_reading_rate
FROM capstone_5iot.gold.mart_sensor_daily
WHERE activity_date BETWEEN :start_date AND :end_date
GROUP BY activity_date
ORDER BY activity_date;
```

**Step 6 – Validate.** Ask the 8 sample questions and compare the answers with `gold.uat_signoff` and the dashboard. If an answer is wrong, add an instruction or a trusted query for it.

## Part 2 – Fixes applied to earlier notebooks on Day 5

**Fix 1 – Azure SQL user name from the secret scope** (`1_azure_dataload`, setup cell)

The password already came from the secret scope. The user name was typed into the code. Store it as a secret once (Databricks CLI):
```
databricks secrets put-secret iot_scope sql-user      # enter the SQL user name when asked
```
Then change one line in the notebook:
```python
# before
SQL_USER = "ltmadmin"
# after
SQL_USER = dbutils.secrets.get("iot_scope", "sql-user")
```
Use the same scope name as the `sql-password` line in your notebook. Secret values are shown as `[REDACTED]` if printed.

**Fix 2 – Silver no longer overwrites a Bronze table** (`4_day2_silver`, CDC cell)

The simulated CDC feed `bronze_work_order_status_cdc` was rewritten on every Silver run. Bronze should be raw and immutable, so it is now written **only once** (when it does not exist yet). After that, Silver only reads it.
```python
# before
save(cdc, f"{BRONZE}.bronze_work_order_status_cdc")
cdc = spark.table(f"{BRONZE}.bronze_work_order_status_cdc")

# after
CDC_TABLE = f"{BRONZE}.bronze_work_order_status_cdc"
if not spark.catalog.tableExists(CDC_TABLE):
    save(cdc, CDC_TABLE)          # one-time seed of the simulated source feed
cdc = spark.table(CDC_TABLE)      # from now on Silver only reads Bronze
```
Row counts stay the same (652 CDC rows). Run the job once afterwards and confirm `7_2_final_regression` is still all PASS.

**Fix 3 – Dashboard views moved to Gold**

The `capstone_5iot.dashboard` schema was deleted. All views are now in `capstone_5iot.gold` with a `vw_` prefix (notebook `7_1`).

## Part 3 – Runbook: `capstone5_iot_pipeline`

### Architecture
```
ADLS iot_vol (CSV batches) --Auto Loader--> Bronze  capstone_5iot.iot_schema   raw, append-only, _source_file / _ingestion_ts
Azure SQL (JDBC, secrets) ---------------->          (machine, plant, failure-code masters)
                         --> Silver  capstone_5iot.silver   typed, DQ rules DQ01-DQ12, quarantine, dedup, CDC MERGE, load audit
                         --> Gold    capstone_5iot.gold     dims, facts, mart_sensor_daily (MERGE), vw_ dashboard views, ML risk table
                         --> AI/BI dashboard, Genie space
```

### Job and task order (job `capstone5_iot_pipeline`)

| # | Task | Notebook | Depends on |
|---|---|---|---|
| 1a | bronze_azure_sql | 01_bronze/1_azure_dataload | – |
| 1b | bronze_adls_autoloader | 01_bronze/2_adls_dataload_and_autoloader | – |
| 2 | data_profile | 01_bronze/3_data_profile | 1a, 1b |
| 3 | silver_transform | 02_silver/4_day2_silver | 2 |
| 4 | gold_build | 03_gold/5_day3_gold | 3 |
| 5 | validate | 6_day4_pipeline_validate | 4 |
| 6 | dashboard_refresh | 7_day4_monitoring_refresh | 5 |

* The Gold `vw_` views are live, so they don't need a job task. They always show the latest Gold data.
* Optional: add `7_2_final_regression` as a task after `validate` to keep a regression record per run.
* **Schedule:** daily 06:00 Europe/London (paused until sign-off). **SLA:** full run under 15 min, data freshness under 24 h. Email alerts on failure and on long runs.

### How to monitor
* **Job runs:** Jobs & Pipelines → capstone5_iot_pipeline → Runs.
* **Loads:** `silver.silver_load_audit`, where every row must have `reconciled = true`.
* **Tests:** `gold.test_results` (Day 4) and `gold.regression_results` (Day 5). Filter `status <> 'PASS'` on the latest `run_ts`.
* **Dashboard:** page "Data quality and pipeline health" (`gold.vw_data_quality`).

### Common failures and fixes

| Symptom | Likely cause | Fix |
|---|---|---|
| bronze_azure_sql: JDBC connection error | Azure SQL resuming from auto-pause, or firewall | Retries usually cover it. Otherwise check the firewall and the secret scope, then repair the run |
| bronze_adls_autoloader: schema mismatch | New or changed CSV columns | Check `_rescued_data`, update schema hints, re-run. The checkpoint stops double loading |
| Many rejected records | Bad source batch | Check `silver_rejected_records` by `rule_id` and `_source_file`, re-deliver the file with the source owner. Never edit Bronze |
| validate fails | A test failed | Read the failed checks, fix the cause, repair from the failed task |
| Dashboard shows old data | An upstream task failed | Repair the run, then refresh the dashboard |

### Replay, repair and restore
* **Repair a run:** open the failed run → **Repair run**. Only the failed task and the tasks after it run again.
* **Replay:** Silver and Gold use overwrite/MERGE and were proven idempotent on Day 4, so a full re-run is safe.
* **Restore a table:** `DESCRIBE HISTORY capstone_5iot.gold.<table>` → find the last good version → `RESTORE TABLE capstone_5iot.gold.<table> TO VERSION AS OF <v>`. Never run VACUUM with less than 168 hours retention.

### Adding a new DQ rule
1. Add the rule to `dq_rule_catalogue` in `4_day2_silver`.
2. Implement it with the same `rule_id` in the Silver rule logic.
3. Add a check for it in the Day 4 tests and in `7_2_final_regression` (section 5).
4. Run the job and confirm the load audit is reconciled and the regression is all PASS.

### Access model

| Group | Access |
|---|---|
| data_engineers | ALL PRIVILEGES on catalog `capstone_5iot` |
| analysts | USE CATALOG, USE SCHEMA and SELECT on `gold` (tables and `vw_` views). No Bronze or Silver |
| dq_stewards | SELECT on `silver.silver_rejected_records` and `silver.dq_rule_catalogue` |

Credentials come only from the secret scope.

### Contacts and ownership

| Role | Name | Contact |
|---|---|---|
| Technical lead | `<name>` | `<email>` |
| Bronze / ingestion owner | `<name>` | `<email>` |
| Silver / DQ owner | `<name>` | `<email>` |
| Gold, dashboard and ML owner | `<name>` | `<email>` |
| Client business owner | `<name>` | `<email>` |

## Part 4 – Key numbers for the presentation
Run this cell and copy the numbers into the slides.

In [0]:
from pyspark.sql import functions as F

GOLD, SILVER = "capstone_5iot.gold", "capstone_5iot.silver"

def q(sql):
    return spark.sql(sql).first()[0]

h = spark.table(f"{GOLD}.vw_kpi_headline").first()
audit = {r.bronze_table: r for r in spark.table(f"{SILVER}.silver_load_audit").collect()}
vital = [r.failure_code for r in spark.table(f"{GOLD}.vw_failure_pareto")
                                      .filter("pareto_class LIKE 'Vital%'").orderBy("pareto_rank").collect()]
top = spark.table(f"{GOLD}.vw_machine_reliability").orderBy(F.desc("downtime_minutes"), "machine_id").first()
region = (spark.table(f"{GOLD}.vw_plant_summary").groupBy("region")
               .agg(F.sum("downtime_minutes").alias("d")).orderBy(F.desc("d")).first())
lift = q(f"SELECT lift_vs_no_failure_machines FROM {GOLD}.vw_failure_correlation WHERE reading_group LIKE '1.%'")
rejects = {r.rule_id: r.n for r in spark.sql(f"SELECT rule_id, count(*) n FROM {SILVER}.silver_rejected_records GROUP BY 1").collect()}
s = audit["bronze_sensor_readings"]

numbers = [
    ("Period", f"{h.period_start} to {h.period_end} ({h.days_in_period} days)"),
    ("Machines / plants / failure codes", f"{h.machines_monitored} / {spark.table(f'{GOLD}.dim_plant').count()} / {spark.table(f'{GOLD}.dim_failure_code').count()}"),
    ("Sensor readings Bronze -> Silver", f"{s.bronze_rows:,} = {s.silver_rows:,} + {s.duplicates_removed} duplicates + {s.rejected} rejects"),
    ("Rejected by rule", ", ".join(f"{k} = {v}" for k, v in sorted(rejects.items()))),
    ("Overall availability", f"{h.overall_availability:.4%}"),
    ("Abnormal reading rate", f"{h.abnormal_reading_rate:.2%}"),
    ("Downtime / failures", f"{h.total_downtime_minutes:,} min / {h.failure_count}"),
    ("MTBF / MTTR", f"{h.mtbf_hours:,} h / {h.mttr_downtime_minutes_per_failure} min per failure"),
    ("Vital few failure codes (80% of downtime)", f"{len(vital)} codes: {', '.join(vital)}"),
    ("Top machine by downtime", f"{top.machine_id} ({top.downtime_minutes} min)"),
    ("Top region by downtime", f"{region.region} ({region.d:,} min)"),
    ("Abnormal lift 24h before failures", str(lift)),
]
display(spark.createDataFrame(numbers, "item string, value string"))

### Presentation outline (12 slides)
1. **Business problem** – unplanned downtime, goal of trusted KPIs and an early-warning pilot. Period, machines and plants.
2. **Architecture** – ADLS + Azure SQL → Auto Loader / JDBC → Bronze → Silver → Gold → dashboard, Genie, MLflow. Unity Catalog, Workflows job.
3. **Data sources and grain** – 6 sources, business keys, row counts.
4. **DQ findings and quarantine** – 12 rules, rejects by rule, Bronze = Silver + duplicates + rejects, telemetry gaps.
5. **Silver design** – typing, dedup, flags, CDC MERGE (late or out-of-order changes cannot overwrite newer status).
6. **Gold model (ERD)** – dims, 3 facts, daily mart with MERGE, surrogate keys, `vw_` dashboard views.
7. **KPIs** – availability, abnormal rate, downtime, MTBF / MTTR, Pareto, top machine and region, lift before failures.
8. **Testing and replay** – Day 4 test matrix, Day 5 regression, idempotent replay, RESTORE recovery, UAT 12/12.
9. **Performance and SLA** – OPTIMIZE / ZORDER benchmark, job duration vs 15-min SLA.
10. **Security and governance** – groups and grants, secrets (Day 5 fix), Bronze immutable (Day 5 fix).
11. **Dashboard demo and ML extension** – 7 dashboard pages, Genie, MLflow runs, registered model with alias `champion`, honest limitations.
12. **Lessons and next steps** – full-period failure log before ML claims, incremental Silver for large volumes, create groups and apply grants, unpause the schedule.

## Part 5 – Day 5 evidence summary
Every value is read from the tables, MLflow and Unity Catalog when the cell runs.

In [0]:
import mlflow
from mlflow import MlflowClient

MODEL_NAME = f"{GOLD}.failure_risk_model"

views = q("""SELECT count(*) FROM capstone_5iot.information_schema.tables
             WHERE table_schema = 'gold' AND table_type = 'VIEW' AND table_name LIKE 'vw_%'""")
dashboard_schema_gone = spark.sql("SHOW SCHEMAS IN capstone_5iot LIKE 'dashboard'").count() == 0

reg = spark.sql(f"""SELECT count(*) AS total, count_if(status = 'PASS') AS passed FROM {GOLD}.regression_results
                     WHERE run_ts = (SELECT max(run_ts) FROM {GOLD}.regression_results)""").first()
uat = spark.sql(f"SELECT count(*) AS total, count_if(status = 'PASS') AS passed FROM {GOLD}.uat_signoff").first()

client = MlflowClient(registry_uri="databricks-uc")
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
metrics = mlflow.get_run(champion.run_id).data.metrics
risk_rows = q(f"""SELECT count(*) FROM {GOLD}.machine_failure_risk
                  WHERE score_date = (SELECT max(score_date) FROM {GOLD}.machine_failure_risk)""")

evidence = [
    ("Dashboard views in Gold", str(views), "9 dashboard views + vw_machine_failure_risk"),
    ("Old dashboard schema deleted", "yes" if dashboard_schema_gone else "NO", "views moved to capstone_5iot.gold"),
    ("Final regression", f"{reg.passed} of {reg.total} PASS", "gold.regression_results, latest run"),
    ("UAT scenarios", f"{uat.passed} of {uat.total} PASS", "gold.uat_signoff"),
    ("Model in Unity Catalog", f"{MODEL_NAME} v{champion.version} @champion", "MLflow experiment capstone05_failure_risk"),
    ("Model holdout metrics", " / ".join(f"{k.replace('holdout_', '')} {metrics.get(k, float('nan')):.3f}"
                                         for k in ["holdout_precision", "holdout_recall", "holdout_f1", "holdout_roc_auc"]),
     "near-chance ROC-AUC, pilot only"),
    ("Machines scored", str(risk_rows), "gold.machine_failure_risk, latest score date"),
]
display(spark.createDataFrame(evidence, "evidence string, result string, detail string"))

## Part 6 – Project sign-off

**Delivered**
* Bronze → Silver → Gold lakehouse in Unity Catalog `capstone_5iot`, run by the Workflows job `capstone5_iot_pipeline` (retries, timeouts, alerts, SLA).
* 12 DQ rules with quarantine and full traceability, CDC protected against out-of-order changes, an idempotent incremental mart.
* Day 4 test matrix, replay and recovery evidence, performance benchmark. Day 5 final regression and 12 UAT scenarios.
* 10 dashboard views in Gold, a 7-page AI/BI dashboard and a Genie space with trusted SQL.
* Pilot failure-risk model in MLflow, registered in Unity Catalog with alias `champion`, scored into `gold.machine_failure_risk` (not production-ready).
* Day 5 fixes: SQL user from secrets, Bronze no longer written by Silver, dashboard views moved to Gold.
* Runbook and presentation outline.

**Open items before go-live**
1. An account admin creates the groups `data_engineers`, `analysts` and `dq_stewards`. Then the Day 4 GRANT cell is run.
2. Get a full-period failure log before any real use of the risk model.
3. Unpause the 06:00 schedule after sign-off.

**Known limitations:** synthetic data. Failure events only cover 2026-07-01 → 2026-07-02.

| Role | Name | Signature | Date |
|---|---|---|---|
| Client business owner | | | |
| Technical lead | | | |
| QA / UAT lead | | | |